# Project 3: Recommender System
#### Group 7: Maura Schorr, Jacob Anderson

In [1]:
import os
import time
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import ParameterGrid
from surprise import Dataset, Reader, NormalPredictor, SVD, accuracy, KNNBasic
from surprise.model_selection import KFold, GridSearchCV

## Data

The MovieLens dataset is a collection of 32 million movie ratings from a research group called GroupLens at the University of Minnesota, and the particular, smaller variant we are using for this analysis is 100k ratings over 9,000 different films by approximately 600 users. Each algorithm uses the same data partitioning to perform a 5-fold cross validation over the data, which ensures for the analysis that any observed differences in RMSE and MAE reflect differences between the algorithms' amenability to the data and task.

Contiguous index maps on the MovieLens user and movie IDs are produced and referenced by the neural network's embedding layers because the raw MovieLens movie IDs are non-contiguous and range up to 193,609, so direct use of these indices would allocate unnecessarily large matrices; the remapping reduces the matrix size in accordance with the number of users and movies.

In [2]:
notebook_dir = os.getcwd()
data_dir = os.path.join(notebook_dir, "..", "data")
output_dir = os.path.join(notebook_dir, "..", "outputs")
os.makedirs(output_dir, exist_ok=True)

ratings_df = pd.read_csv(os.path.join(data_dir, "ratings.csv"))
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Cross-Validation Utilities

In [3]:
def build_surprise_dataset(ratings_df):
    reader = Reader(rating_scale=(0.5, 5.0))
    return Dataset.load_from_df(ratings_df[["userId", "movieId", "rating"]], reader)


# These are the 5 shared folds that need to be used by all algorithms
def build_folds(dataset):
    kf = KFold(n_splits=5, random_state=42, shuffle=True)
    return list(kf.split(dataset))


# Contiguous index maps for NN embedding layers
def build_id_maps(ratings_df):
    user_ids = sorted(ratings_df["userId"].unique())
    movie_ids = sorted(ratings_df["movieId"].unique())
    user_map = {uid: i for i, uid in enumerate(user_ids)}
    movie_map = {mid: i for i, mid in enumerate(movie_ids)}
    return user_map, movie_map


dataset = build_surprise_dataset(ratings_df)
folds = build_folds(dataset)
user_map, movie_map = build_id_maps(ratings_df)

In [4]:
# Converts a Surprise trainset to numpy arrays
def trainset_to_arrays(trainset, user_map, movie_map):
    users, movies, ratings = [], [], []
    for u, i, r in trainset.all_ratings():
        users.append(user_map[int(trainset.to_raw_uid(u))])
        movies.append(movie_map[int(trainset.to_raw_iid(i))])
        ratings.append(r)
    return (np.array(users), np.array(movies), np.array(ratings, dtype=np.float32))


# Converts a Surprise testset to numpy arrays
def testset_to_arrays(testset, user_map, movie_map):
    users, movies, ratings = [], [], []
    for uid, iid, r in testset:
        users.append(user_map[int(uid)])
        movies.append(movie_map[int(iid)])
        ratings.append(r)
    return (np.array(users), np.array(movies), np.array(ratings, dtype=np.float32))


def compute_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


def compute_mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def save_predictions(predictions_df, filename):
    predictions_df.to_csv(os.path.join(output_dir, filename), index=False)


def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return (f"{h}:{m:02d}:{s:02d}")


def report_cv_results(fold_results):
    avg_rmse = np.mean([r["rmse"] for r in fold_results])
    avg_mae = np.mean([r["mae"] for r in fold_results])
    avg_time = np.mean([r["time"] for r in fold_results])
    print(f"{'Fold':<8} {'RMSE':>8} {'MAE':>8} {'Time':>10}")
    for r in fold_results:
        print(f"{r['fold']:<8} {r['rmse']:>8.4f} {r['mae']:>8.4f} {format_time(r['time']):>10}")
    print(f"{'Mean':<8} {avg_rmse:>8.4f} {avg_mae:>8.4f} {format_time(avg_time):>10}")

## Algorithms

### Random

The random algorithm is a baseline upon which to compare the other algorithms. In our analysis, this baseline uses the Surprise library's `NormalPredictor` impelementation, which fits a Gaussian distribution to the training ratings and samples predicted values for each test pair.

Formally, given a training set $\mathcal{R} = \{r_{ui}\}$ of ratings for users $ u $ and movie $ i $, parameters for the normal distribution are fit to the empirical distribution using maximum likelihood: 

$$\mu = \frac{1}{|\mathcal{R}|} \sum_{(u,i) \in \mathcal{R}} r_{ui}$$

$$\sigma^2 = \frac{1}{|\mathcal{R}|} \sum_{(u,i) \in \mathcal{R}} (r_{ui} - \mu)^2$$

Note that assumption of the distribution is an approximation due to the nature of the MovieLens ratings being disrete and bounded from 0.5 to 5.0.

Then, for each test pair $(u, i) \in \mathcal{T}$, a predicted rating is sampled from the distribution

$$\hat{r}_{ui} \sim \mathcal{N}(\mu, \sigma^2)$$

and the sample is then constrained to the valid range for ratings:

$$\hat{r}_{ui} \leftarrow \text{clip}(\hat{r}_{ui},\ 0.5,\ 5.0)$$

In [5]:
def random_cv(folds):
    fold_results = []
    all_predictions = []

    for fold_i, (trainset, testset) in enumerate(folds):
        algo = NormalPredictor()

        start = time.time()
        algo.fit(trainset)
        np.random.seed(42)
        predictions = algo.test(testset)
        elapsed = time.time() - start

        rmse = accuracy.rmse(predictions, verbose=False)
        mae = accuracy.mae(predictions, verbose=False)

        fold_results.append({
            "fold": fold_i + 1,
            "rmse": rmse,
            "mae": mae,
            "time": elapsed,
        })

        all_predictions.extend([
            {
                "userId": int(pred.uid),
                "movieId": int(pred.iid),
                "predicted_rating": pred.est,
            }
            for pred in predictions
        ])

    return fold_results, pd.DataFrame(all_predictions)

In [6]:
random_fold_results, random_predictions = random_cv(folds)
report_cv_results(random_fold_results)
save_predictions(random_predictions, "random_predictions.csv")

Fold         RMSE      MAE       Time
1          1.4278   1.1399    0:00:00
2          1.4282   1.1346    0:00:00
3          1.4184   1.1322    0:00:00
4          1.4342   1.1472    0:00:00
5          1.4358   1.1485    0:00:00
Mean       1.4289   1.1405    0:00:00


The random cross-validation yielded a mean RMSE of 1.4289 and mean MAE of 1.1405. These performance measures are expectedly consistent across folds since the predictions are conditioned on nothing related to the characteristics of hte users nor the items and considers only the empirical rating distribution.


### kNN

In [7]:
#defining knn model 

#tested k=4, 8, 10, 20, 40, 80, 100, 120 and after 80 the rmse and mae didnt improve much and the runtimes started increasing slightly so going with k=80 
def run_knn_cv(folds, k=80, user_based=False):
    fold_results = []
    all_predictions = []

    for fold_num, (trainset, testset) in enumerate(folds, start=1):
        sim_options = {
            "name": "cosine",
            "user_based": user_based #it compares movies to movies not users to users 
        }

        model = KNNBasic(
            k=k,
            min_k=1,
            sim_options=sim_options,
            verbose=False
        )

        #track the time it runs on each fold 
        start_time = time.time()

        model.fit(trainset)
        predictions = model.test(testset)

        end_time = time.time()
        elapsed_time = end_time - start_time

        #stats for the run on each fold 
        rmse = accuracy.rmse(predictions, verbose=False)
        mae = accuracy.mae(predictions, verbose=False)


        fold_results.append({
            "fold": fold_num,
            "k": k,
            "RMSE": rmse,
            "MAE": mae,
            "Time": elapsed_time
        })

        for pred in predictions:
            all_predictions.append({
                "fold": fold_num,
                "userId": pred.uid,
                "movieId": pred.iid,
                "actual_rating": pred.r_ui,
                "predicted_rating": pred.est,
                "algorithm": "KNN",
                "k": k
            })

    results_df = pd.DataFrame(fold_results)
    predictions_df = pd.DataFrame(all_predictions)


    avg_rmse = results_df["RMSE"].mean()
    avg_mae = results_df["MAE"].mean()
    avg_time = results_df["Time"].mean()

    return results_df, predictions_df, avg_rmse, avg_mae, avg_time



In [8]:
#create the final predictions csv 
knn_fold_results, final_knn_predictions, avg_rmse, avg_mae, avg_time = run_knn_cv(
    folds=folds,
    k=80,
    user_based=False,
)

#putting my results into the format so the report_cv_results can work with my knn 
knn_fold_results_for_report = [
    {
        "fold": int(row["fold"]),
        "rmse": row["RMSE"],
        "mae": row["MAE"],
        "time": row["Time"]
    }
    for _, row in knn_fold_results.iterrows()
]

report_cv_results(knn_fold_results_for_report)
knn_preds_relevant_columns = final_knn_predictions[["userId", "movieId", "predicted_rating"]]
save_predictions(knn_preds_relevant_columns, "knn_predictions.csv" )


Fold         RMSE      MAE       Time
1          0.9716   0.7541    0:00:09
2          0.9585   0.7454    0:00:08
3          0.9544   0.7435    0:00:08
4          0.9703   0.7554    0:00:08
5          0.9675   0.7505    0:00:08
Mean       0.9645   0.7498    0:00:08


##### Determining hyperparameters 
 
We test k = 4, 8, 10, 20, 40, 80, 100 and 120 to determine the k that returned the best RMSE and MAE on average 

In [9]:
#code that I ran to determine the best k values 
#testing different k values 

#k values i want to test 
k_values = [4, 8, 10, 20, 40, 80, 100, 120]

#holds the results from the run with each k value 
summary_results = []

for k in k_values:

    fold_results, predictions, avg_rmse, avg_mae, avg_time = run_knn_cv(
        folds=folds,
        k=k,
        user_based=False,
    )

    #save information from the runs with each k 
    summary_results.append({
        "k": k,
        "Average RMSE": avg_rmse,
        "Average MAE": avg_mae,
        "Average Time(sec)": avg_time
    })

knn_summary = pd.DataFrame(summary_results)
knn_summary = knn_summary.sort_values("Average RMSE")

knn_summary
#going to use k=80 because increasing the k after that doesn't significantly improve the RMSE or MAE after that and the time is similar 


,k,Average RMSE,Average MAE,Average Time(sec)
7,120,0.958616,0.744324,8.318944
6,100,0.961176,0.746734,8.147541
5,80,0.964469,0.749778,7.964877
4,40,0.977247,0.760655,7.192068
3,20,0.996522,0.776532,6.892623
2,10,1.025997,0.800367,6.835524
1,8,1.040935,0.811983,6.685279
0,4,1.106903,0.861870,6.784323


### SVD

The SVD algorithm decompases the user-item (UI) matrix into latent user and item factors that are estimated using stochastic gradient descent (SGD). Formally, the prediction for each user-item rating is given as a linear combination of the latent factor vectors and learned bias terms for each item and user:

$$\hat{r}_{ui} = \mu + b_u + b_i + q_i^T p_u$$

where $ \mu $ is the empirical mean rating, $ b_u $ and $ b_i $ are the learned bias terms for each user and item, and $ p_u \in \mathbb{R}^k $ and $ q_i \in \mathbb{R}^k $ dotted together represents the interaction between the user's latent characteristics and the item's latent characteristics. Generally speaking, the bias terms themselves represented the tendency of some user to rating above or below the empirical average rating and the tendency of some item to be rated above or below the empirical average rating, respectively.

These parameters $ \{b_u, b_i, p_u, q_i\} $ are estimated by minimizing the regularized squared error over the ratings in the training set:

$$\min_{b_u, b_i, p_u, q_i} \sum_{(u,i) \in \mathcal{R}} \left(r_{ui} - \hat{r}_{ui}\right)^2 + \lambda\left(b_u^2 + b_i^2 + \|p_u\|^2 + \|q_i\|^2\right)$$

where $ \lambda $ represents the regularization strength controlling the penalty on the parameter magnitude as a means to prevent overfitting on the data. To find a local optimum for this objective, a prediction error is produced for each rating as:

$$e_{ui} = r_{ui} - \hat{r}_{ui}$$

and the parameters are updated accordingly:

$$ b_u \leftarrow b_u + \gamma \left(e_{ui} - \lambda b_u\right) $$

$$ b_i \leftarrow b_i + \gamma \left(e_{ui} - \lambda b_i\right) $$

$$ p_u \leftarrow p_u + \gamma \left(e_{ui} \cdot q_i - \lambda p_u\right) $$

$$ q_i \leftarrow q_i + \gamma \left(e_{ui} \cdot p_u - \lambda q_i\right) $$

where $ \gamma $ is the learning rate hyperparameter. Each update is applied in series for each of the examples in the training set to constitue one epoch, and in this analysis, this algorithm is repeated over 20 epochs to achieve convergence.

A final predicted rating for each test pair $ (u, i) \in \mathcal{T} $ again is given by $ \hat{r}_{ui} = \mu + b_u + b_i + q_i^T p_u $, which is performed for each fold to produce predicted ratings across the entire dataset.


#### Hyperparameter Search

This algorithm is dicated by four hyperparameters, the choice of latent factor dimension $ k $, the number of epochs, the learning rate $ \gamma $, and the regularization strength $ \lambda $. $ k $ controls the dimensionality of the latent space and balances the benefit of a more complex item user interaction with risk of overfitting to the dataset given its size, $ \gamma $ controls the step size of each SGD update and will determine how efficiently the SGD can converge toward an optimum, and the setting of $ \lambda $ balances the cost of overfitting with potentially underfitting with the regularization is too strong.

These hyperparameter were set for the cross validation analysis by a search over the following space, using average RMSE across folds as the objective heuristic:

$$ k \in \{50, 100, 150, 200\} $$

$$ \text{epochs} \in \{20, 50, 100\} $$

$$ \gamma \in \{0.002, 0.005, 0.01\} $$

$$ \lambda \in \{0.02, 0.05, 0.1\} $$

In [10]:
hyperparameter_space = {
    "n_factors": [50, 100, 150, 200],
    "n_epochs": [20, 50, 100],
    "lr_all": [0.002, 0.005, 0.01],
    "reg_all": [0.02, 0.05, 0.1]
}


def search_svd_params(dataset):
    gs = GridSearchCV(SVD, {**hyperparameter_space, "random_state": [42]}, measures=["rmse"], cv=5, n_jobs=1)
    gs.fit(dataset)
    best_params = gs.best_params["rmse"]
    print(f"Optimal RMSE: {gs.best_score['rmse']:.4f}")
    print(f"Optimal parameters: {best_params}")
    return best_params

In [11]:
svd_best_params = search_svd_params(dataset)

Optimal RMSE: 0.8485
Optimal parameters: {'n_factors': 200, 'n_epochs': 100, 'lr_all': 0.01, 'reg_all': 0.1, 'random_state': 42}


The optimal parameter settings given by the search are $k = 200$, $\text{epochs} = 100$, $\gamma = 0.01$, and $\lambda = 0.1$. The search suggest that the model benefits from the largest latent representation to capture user and item characteristics, which in turn requires a largest number of epochs with which to produce a model of the user-item interactions. The high learning rate may indicate that the optimization space is amenable to largest parameter updates, and the strong regularization is likely necessary to prevent the model from overfitting the train set given how large the latent representations are.

In [12]:
def svd_cv(folds, n_factors, n_epochs, lr_all, reg_all):
    fold_results = []
    predictions = []

    for fold_i, (trainset, testset) in enumerate(folds):
        algo = SVD(
            n_factors=n_factors,
            n_epochs=n_epochs,
            lr_all=lr_all,
            reg_all=reg_all,
            random_state=42,
        )

        start = time.time()
        algo.fit(trainset)
        fold_predictions = algo.test(testset)
        elapsed = time.time() - start

        rmse = accuracy.rmse(fold_predictions, verbose=False)
        mae = accuracy.mae(fold_predictions, verbose=False)

        fold_results.append({
            "fold": fold_i + 1,
            "rmse": rmse,
            "mae": mae,
            "time": elapsed,
        })

        predictions.extend([
            {
                "userId": int(pred.uid),
                "movieId": int(pred.iid),
                "predicted_rating": pred.est,
            }
            for pred in fold_predictions
        ])

    return fold_results, pd.DataFrame(predictions)

In [13]:
svd_fold_results, svd_predictions = svd_cv(
    folds,
    n_factors=svd_best_params["n_factors"],
    n_epochs=svd_best_params["n_epochs"],
    lr_all=svd_best_params["lr_all"],
    reg_all=svd_best_params["reg_all"],
)
report_cv_results(svd_fold_results)
save_predictions(svd_predictions, "svd_predictions.csv")

Fold         RMSE      MAE       Time
1          0.8559   0.6536    0:00:03
2          0.8439   0.6458    0:00:03
3          0.8372   0.6419    0:00:03
4          0.8450   0.6477    0:00:03
5          0.8538   0.6538    0:00:03
Mean       0.8472   0.6486    0:00:03


Cross-validation on SVD produced an average RMSE of 0.8472 and average MAE of 0.6486. The performance measures across folds are consistent and exhibit an significant improvement of 0.58 RMSE units relative to the random algorithm. 

### Matrix Factorization

In [ ]:
#defining matrix factorization with regularization 
#going to use surprise's svd but tweak some of the parameters 


SVD( #just including this to keep parameters straight for my knowledge 
    n_factors=50,      #hidden dimensions for each movie/user
    n_epochs=20,       #number of times the model loops over the training data 
    lr_all=0.005,      #learning rate- how aggressive model adjusts after each mistake 
    reg_all=0.02,      #regularization- penalizes large values in user/movie vector
    biased=False        #Does not take biases into account 
)

#best number of factors/regularization combo was 100 factors adn 0.05 regularization 
def run_mf_reg_cv(folds, n_factors=100, reg_all=0.05, n_epochs=20, lr_all=0.005):
    fold_results = []
    all_predictions = []

    for fold_num, (trainset, testset) in enumerate(folds, start=1):

        model = SVD(
            n_factors=n_factors,
            n_epochs=n_epochs,
            lr_all=lr_all,
            reg_all=reg_all,
            biased=False,
            random_state=42
        )

        start_time = time.time()

        model.fit(trainset)
        predictions = model.test(testset)

        end_time = time.time()
        elapsed_time = end_time - start_time

        rmse = accuracy.rmse(predictions, verbose=False)
        mae = accuracy.mae(predictions, verbose=False)

        fold_results.append({
            "fold": fold_num,
            "n_factors": n_factors,
            "reg_all": reg_all,
            "n_epochs": n_epochs,
            "RMSE": rmse,
            "MAE": mae,
            "Time": elapsed_time
        })

        for pred in predictions:
            all_predictions.append({
                "userId": pred.uid,
                "movieId": pred.iid,
                "predicted_rating": pred.est
            })

    results_df = pd.DataFrame(fold_results)
    predictions_df = pd.DataFrame(all_predictions)

    avg_rmse = results_df["RMSE"].mean()
    avg_mae = results_df["MAE"].mean()
    avg_time = results_df["Time"].mean()
        
    return results_df, predictions_df, avg_rmse, avg_mae, avg_time


In [ ]:
#final run 
#testing different n_factors and reg_all were testd and n_factors=10 and reg_all=0.02 is the best combo 
final_mf_results, final_mf_predictions, mf_rmse, mf_mae, mf_time = run_mf_reg_cv(
    folds=folds,
    n_factors=10,
    reg_all=0.02,
    n_epochs=20,
    lr_all=0.005,
)

#putting my results into the format so the report_cv_results can work with my knn 
mf_fold_results_for_report = [
    {
        "fold": int(row["fold"]),
        "rmse": row["RMSE"],
        "mae": row["MAE"],
        "time": row["Time"]
    }
    for _, row in final_mf_results.iterrows()
]


report_cv_results(mf_fold_results_for_report)
mf_preds_relevant_columns = final_mf_predictions[["userId", "movieId", "predicted_rating"]]
save_predictions(mf_preds_relevant_columns, "mf_predictions.csv" )

save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../outputs
Fold         RMSE      MAE       Time
1          0.9465   0.7234    0:00:00
2          0.9378   0.7175    0:00:00
3          0.9434   0.7199    0:00:00
4          0.9285   0.7086    0:00:00
5          0.9502   0.7218    0:00:00
Mean       0.9413   0.7183    0:00:00


#### Choosing Hyperparameters 

In [16]:
#testing parameters for mf 
mf_results = []

#only going to test n factors and reg all parameters 
#reg_all -> smaller higher risk of over fitting, larger risk of under fitting 
for n_factors in [10, 20, 50, 80, 100]:
    for reg_all in [0.01, 0.02, 0.05, 0.07, 0.1]:

        fold_results, predictions, avg_rmse, avg_mae, avg_time = run_mf_reg_cv(
            folds=folds,
            n_factors=n_factors,
            reg_all=reg_all,
            n_epochs=20,
            lr_all=0.005,
        )

        mf_results.append({
            "n_factors": n_factors,
            "reg_all": reg_all,
            "Average RMSE": avg_rmse,
            "Average MAE": avg_mae,
            "Average Time": avg_time
        })

mf_summary = pd.DataFrame(mf_results)
mf_summary

save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../outputs
save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../outputs
save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../outputs
save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../outputs
save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../outputs
save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../outputs
save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../outputs
save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../outputs
save_predictions = True
output_dir = /home/jande/csds435_project3_group7/recommender-system/notebook/../

,n_factors,reg_all,Average RMSE,Average MAE,Average Time
0,10,0.01,0.942346,0.719496,0.288756
1,10,0.02,0.941278,0.718258,0.284370
2,10,0.05,0.952464,0.726322,0.293748
3,10,0.07,0.963082,0.734600,0.280653
4,10,0.10,0.979012,0.747428,0.280945
5,20,0.01,0.946553,0.725627,0.320466
6,20,0.02,0.943112,0.722071,0.305317
7,20,0.05,0.951971,0.727200,0.300461
8,20,0.07,0.962345,0.734870,0.349338
9,20,0.10,0.978287,0.747411,0.299972


### Neural Network

The deep neural network (dNN) algorithm implements an embedding-based collaborative filtering. Users and movie are represented by learned vectors in a shared latent space which are concatenated and passed through a neural network structure to produce a rating prediction for a given user-item interaction.

First, the MovieLens user and movie IDs are remapped as described in the Data section, and from these IDs embeddings are retrived from the lookup tables at the beginning of the network:

$$ e_u = E_U[u] \in \mathbb{R}^d, \quad e_i = E_I[i] \in \mathbb{R}^d $$

These embeddings are learned just like the other weight matrices in the network via backpropagation. These two embeddings are concatenated: $ x_{ui} = [e_u \,\|\, e_i] \in \mathbb{R}^{2d} $, and given as the input to the primary network structure.

First, a dropout layer with rate $ p $ is applied and the result is passed to a sequence of fully-connected layers, with each followed by a rectified linear unit (ReLU) activation and another dropout layer with the same rate hyperparameter:

$$h^{(0)} = \text{Dropout}(x_{ui},\ p)$$

$$h^{(\ell)} = \text{Dropout}(\text{ReLU}(W^{(\ell)} h^{(\ell-1)} + b^{(\ell)}),\ p), \quad \ell = 1, \ldots, L$$

The representation is then passed through a linear layer and a sigmoid activation that is finally scaled to the valid range of MovieLens ratings as $ \hat{r}_{ui} = 0.5 + 4.5 \cdot \sigma(W^{(L+1)} h^{(L)} + b^{(L+1)}) $.

The network is trained to minimize MSE over each training fold and parameter updates over the training are performed using Adam optimization. The updates themselves are faciliated by mini-batch SGD, where loss computed over each batch is of ratings $\mathcal{B}$ is given as:

$$L = \frac{1}{|B|} \sum_{(u,i) \in B} (r_{ui} - \hat{r}_{ui})^2$$

The gradients $ L $ with respect to all parameters are computed via backpropagation and the Adam update is applied. This process repeats over all batches in the training set constituting one epoch, which is repeated for a selected number of epochs.

Once parameter estimations have been produced, a predicted rating for each test pair $(u, i) \in \mathcal{T}$ is given by a forward pass through the network, and this process is repeated for each fold.


#### Hyperparameter Search

The hyperparameters that dictate this model can be split into two classes: those that related to the network structure and those that relate to the training of the network. Regarding the network structure, the embedding dimension of the users and items controls how complex the interactions of latent representations may be, relative to the scale of the dataset, and the hidden layer configuration controls the depth and width of the MLP applied to the embeddings, which may be more or less amenable depending on overfitting constraints and how well different configurations and capture patterns of interactions between the latent item and user representations. With a optimal architecture for hte given task, selections of $ \alpha $ will control the ability of the network to converge to learning optimums, an intelligently selected dropout rate $ p $ will help mitigate complex weight dependencies in the network that could constrain optimal convergence of the architecture, and $ |B| $ will balance the noise/instability of smaller gradient updates with the ability to potentially escape local minima and achieve better generalization over the training set.

Hyperparameter selection for the neural network is performed in two stages. In each stage, each selection in the search space produces an average RMSE across folds in a 5-fold cross-validation, and the optimally minimal selection is used for the cross validation. The first stage of the search selects an embedding dimesions $ d $ and a hidden layer structure over the following search space, with training settings fixed at $\alpha = 0.001$, $p = 0.2$, and $ |B| = 256 $:

$$d \in \{32, 64, 128\}$$

$$\text{hidden layers} \in \{[128, 64],\ [256, 128],\ [256, 128, 64]\}$$

The second stage searches for optimal settings for $ \alpha $, $ p $, and $ |B| $ in the following space for the optimal network architecture found in the first search:

$$\alpha \in \{0.001, 0.005, 0.01\}$$

$$p \in \{0.1, 0.2, 0.4\}$$

$$|B| \in \{128, 256, 512\}$$

#### Architecture Search

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed during search, carried into cross-validation
epochs = 20
learning_rate = 0.001
dropout = 0.2
batch_size = 256


class EmbeddingNet(nn.Module):
    def __init__(self, n_users, n_movies, embedding_dim, hidden_layers, dropout):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.movie_embedding = nn.Embedding(n_movies, embedding_dim)
        self.embedding_dropout = nn.Dropout(dropout)

        layers = []
        input_dim = embedding_dim * 2

        for hidden_dim in hidden_layers:
            layers += [nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
            input_dim = hidden_dim

        layers.append(nn.Linear(input_dim, 1))

        self.hidden = nn.Sequential(*layers)
        self.min_rating = 0.5
        self.max_rating = 5.0

    def forward(self, user_ids, movie_ids):
        user_emb = self.user_embedding(user_ids)
        movie_emb = self.movie_embedding(movie_ids)
        x = self.embedding_dropout(torch.cat([user_emb, movie_emb], dim=1))

        # Scaled to the rating range
        return self.min_rating + (self.max_rating - self.min_rating) * torch.sigmoid(self.hidden(x).squeeze(1))


def make_tensors(users, movies, ratings):
    return (
        torch.tensor(users, dtype=torch.long),
        torch.tensor(movies, dtype=torch.long),
        torch.tensor(ratings, dtype=torch.float32),
    )


def train_and_evaluate(n_users, n_movies, train_users, train_movies, train_ratings,
                       test_users, test_movies, test_ratings,
                       embedding_dim, hidden_layers, dropout, lr, batch_size, n_epochs):
    torch.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    model = EmbeddingNet(n_users, n_movies, embedding_dim, hidden_layers, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_dataset = TensorDataset(*make_tensors(train_users, train_movies, train_ratings))

    generator = torch.Generator()
    generator.manual_seed(42)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=generator)

    model.train()
    for _ in range(n_epochs):
        for batch_users, batch_movies, batch_ratings in train_loader:
            batch_users = batch_users.to(device)
            batch_movies = batch_movies.to(device)
            batch_ratings = batch_ratings.to(device)
            optimizer.zero_grad()
            loss_fn(model(batch_users, batch_movies), batch_ratings).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        t_users, t_movies, _ = make_tensors(test_users, test_movies, test_ratings)
        preds = model(t_users.to(device), t_movies.to(device)).cpu().numpy()

    rmse = compute_rmse(test_ratings, preds)
    return rmse, preds

In [18]:
# Architecture parameters, optimizer settings held fixed
architecture_parameter_space = {
    "embedding_dim": [32, 64, 128],
    "hidden_layers": [[128, 64], [256, 128], [256, 128, 64]]
}


# Architecture search over all folds with fixed optimizer settings
def architecture_search(folds, user_map, movie_map, n_users, n_movies):
    best_rmse = float("inf")
    best_params = None

    for params in ParameterGrid(architecture_parameter_space):
        fold_rmses = []

        for trainset, testset in folds:
            train_users, train_movies, train_ratings = trainset_to_arrays(trainset, user_map, movie_map)
            test_users, test_movies, test_ratings = testset_to_arrays(testset, user_map, movie_map)

            rmse, _ = train_and_evaluate(
                n_users, n_movies,
                train_users, train_movies, train_ratings,
                test_users, test_movies, test_ratings,
                embedding_dim=params["embedding_dim"],
                hidden_layers=params["hidden_layers"],
                dropout=dropout,
                lr=learning_rate,
                batch_size=batch_size,
                n_epochs=epochs,
            )

            fold_rmses.append(rmse)

        average_rmse = np.mean(fold_rmses)
        if average_rmse < best_rmse:
            best_rmse = average_rmse
            best_params = params

    print(f"Architecture search:")
    print(f"Optimal RMSE: {best_rmse:.4f}")
    print(f"Optimal parameters: {best_params}")

    return best_params

In [19]:
n_users = len(user_map)
n_movies = len(movie_map)

architecture = architecture_search(folds, user_map, movie_map, n_users, n_movies)

Architecture search:
Optimal RMSE: 0.8765
Optimal parameters: {'embedding_dim': 64, 'hidden_layers': [128, 64]}


The architecture search selected $d = 64$ with hidden layers $[128, 64]$, achieving a mean RMSE of 0.8765. The model appears to benefit from some reduction in the representational capacity and network size, likely due to potential overfitting behavior on the dataset given its scale.

#### Optimizer Search

In [20]:
# Optimizer parameters, architecture parameters are fixed optimally from the previous search
optimizer_parameter_space = {
    "lr": [0.001, 0.005, 0.01],
    "dropout": [0.1, 0.2, 0.4],
    "batch_size": [128, 256, 512]
}


# Optimizer search over all folds with architecture fixed at best from architecture search
def optimizer_search(folds, user_map, movie_map, n_users, n_movies, embedding_dim, hidden_layers):
    best_rmse = float("inf")
    best_params = None

    for params in ParameterGrid(optimizer_parameter_space):
        fold_rmses = []

        for trainset, testset in folds:
            train_users, train_movies, train_ratings = trainset_to_arrays(trainset, user_map, movie_map)
            test_users, test_movies, test_ratings = testset_to_arrays(testset, user_map, movie_map)

            rmse, _ = train_and_evaluate(
                n_users, n_movies,
                train_users, train_movies, train_ratings,
                test_users, test_movies, test_ratings,
                embedding_dim=embedding_dim,
                hidden_layers=hidden_layers,
                dropout=params["dropout"],
                lr=params["lr"],
                batch_size=params["batch_size"],
                n_epochs=epochs,
            )

            fold_rmses.append(rmse)

        average_rmse = np.mean(fold_rmses)
        if average_rmse < best_rmse:
            best_rmse = average_rmse
            best_params = params

    print(f"Optimizer search:")
    print(f"Optimal RMSE: {best_rmse:.4f}")
    print(f"Optimal parameters: {best_params}")

    return best_params

In [21]:
optimizers = optimizer_search(
    folds, user_map, movie_map, n_users, n_movies,
    embedding_dim=architecture["embedding_dim"],
    hidden_layers=architecture["hidden_layers"],
)

Optimizer search:
Optimal RMSE: 0.8672
Optimal parameters: {'batch_size': 128, 'dropout': 0.4, 'lr': 0.001}


The optimizer search identified $\alpha = 0.001$, $p = 0.4$, and $ |B| = 128 $ as the optimal settings for the given architecture, reporting a mean RMSE of 0.8672. The search selected the smallest learning rate and batch size alongside with the highest dropout rate, so the architecture is more performant under conditions where the parameter updates are very stable, small, and heavily constrained so as not the produce dependencies between other weights in the network. The high dropout especially indicates that a network of this scale on this data likely is attempting to memorize the observed ratings, which is corroborated by the smaller batch size helping to prevent convergence to minima that do not generalize well.

In [22]:
def nn_cv(folds, user_map, movie_map, n_users, n_movies,
           embedding_dim, hidden_layers, dropout, lr, batch_size):
    fold_results = []
    predictions = []

    for fold_i, (trainset, testset) in enumerate(folds):
        train_users, train_movies, train_ratings = trainset_to_arrays(trainset, user_map, movie_map)
        test_users, test_movies, test_ratings = testset_to_arrays(testset, user_map, movie_map)

        start = time.time()
        rmse, preds = train_and_evaluate(
            n_users, n_movies,
            train_users, train_movies, train_ratings,
            test_users, test_movies, test_ratings,
            embedding_dim=embedding_dim,
            hidden_layers=hidden_layers,
            dropout=dropout,
            lr=lr,
            batch_size=batch_size,
            n_epochs=epochs,
        )
        elapsed = time.time() - start

        mae = compute_mae(test_ratings, preds)

        fold_results.append({
            "fold": fold_i + 1,
            "rmse": rmse,
            "mae": mae,
            "time": elapsed,
        })

        predictions.extend([
            {
                "userId": int(uid),
                "movieId": int(iid),
                "predicted_rating": float(p),
            }
            for (uid, iid, _), p in zip(testset, preds)
        ])

    return fold_results, pd.DataFrame(predictions)

In [23]:
nn_fold_results, nn_predictions = nn_cv(
    folds, user_map, movie_map, n_users, n_movies,
    embedding_dim=architecture["embedding_dim"],
    hidden_layers=architecture["hidden_layers"],
    dropout=optimizers["dropout"],
    lr=optimizers["lr"],
    batch_size=optimizers["batch_size"],
)
report_cv_results(nn_fold_results)
save_predictions(nn_predictions, "nn_predictions.csv")

Fold         RMSE      MAE       Time
1          0.8733   0.6724    0:00:38
2          0.8661   0.6721    0:00:36
3          0.8612   0.6718    0:00:36
4          0.8618   0.6623    0:00:42
5          0.8736   0.6792    0:00:43
Mean       0.8672   0.6716    0:00:39


The neural network yielded a mean RMSE of 0.8672 and mean MAE of 0.6716, and the performance across folds remains quite consistent. The hyperparameter search does well to address the network's proclivity towards overfitting, but without aggressive dropout rate and a small learning rate the network will tend to memorize the observed ratings over learning a generalized concept, so the model's larger capacity relative to other algorithms is constraining to its performance.

## Evaluation

Root mean squared error (RMSE) and mean absolute error (MAE) are computed between the sample set and the predicted ratings for each algorithm:

$$\text{RMSE} = \sqrt{\frac{1}{|\mathcal{T}|}\sum_{(u,i) \in \mathcal{T}}(r_{ui} - \hat{r}_{ui})^2}$$

$$\text{MAE} = \frac{1}{|\mathcal{T}|}\sum_{(u,i) \in \mathcal{T}}|r_{ui} - \hat{r}_{ui}|$$

The tables below give the performance metrics for each fold across all five algorithms.

In [24]:
print("Random")
report_cv_results(random_fold_results)

print("\nkNN")
report_cv_results(knn_fold_results_for_report)

print("\nSVD")
report_cv_results(svd_fold_results)

print("\nMatrix Factorization") 
report_cv_results(mf_fold_results_for_report)

print("\nNeural Network")
report_cv_results(nn_fold_results)

Random
Fold         RMSE      MAE       Time
1          1.4278   1.1399    0:00:00
2          1.4282   1.1346    0:00:00
3          1.4184   1.1322    0:00:00
4          1.4342   1.1472    0:00:00
5          1.4358   1.1485    0:00:00
Mean       1.4289   1.1405    0:00:00

kNN
Fold         RMSE      MAE       Time
1          0.9716   0.7541    0:00:09
2          0.9585   0.7454    0:00:08
3          0.9544   0.7435    0:00:08
4          0.9703   0.7554    0:00:08
5          0.9675   0.7505    0:00:08
Mean       0.9645   0.7498    0:00:08

SVD
Fold         RMSE      MAE       Time
1          0.8559   0.6536    0:00:03
2          0.8439   0.6458    0:00:03
3          0.8372   0.6419    0:00:03
4          0.8450   0.6477    0:00:03
5          0.8538   0.6538    0:00:03
Mean       0.8472   0.6486    0:00:03

Matrix Factorization
Fold         RMSE      MAE       Time
1          0.9465   0.7234    0:00:00
2          0.9378   0.7175    0:00:00
3          0.9434   0.7199    0:00:00
4          

### Pairwise RMSE Table

In [25]:
random_preds = pd.read_csv(os.path.join(output_dir, "random_predictions.csv"))
knn_preds = pd.read_csv(os.path.join(output_dir, "knn_predictions.csv"))
svd_preds = pd.read_csv(os.path.join(output_dir, "svd_predictions.csv"))
mf_preds = pd.read_csv(os.path.join(output_dir, "mf_predictions.csv"))
nn_preds = pd.read_csv(os.path.join(output_dir, "nn_predictions.csv"))

algos = {
    "random": random_preds,
    "knn": knn_preds,
    "svd": svd_preds,
    "mf": mf_preds,
    "nn": nn_preds,
}

# Merges the predictions on userId and movieId keys and renames predicted_rating columns to algorithm names
merged = None
for name, df in algos.items():
    df = df[["userId", "movieId", "predicted_rating"]].rename(columns={"predicted_rating": name})
    merged = df if merged is None else merged.merge(df, on=["userId", "movieId"])

# Compute pairwise RMSE between predictions
algo_names = list(algos.keys())
pairwise = pd.DataFrame(index=algo_names, columns=algo_names, dtype=float)

for a, b in itertools.product(algo_names, algo_names):
    pairwise.loc[a, b] = np.sqrt(np.mean((merged[a] - merged[b]) ** 2))

pairwise.round(4)

,random,knn,svd,mf,nn
random,0.0000,1.0961,1.1311,1.2291,1.1352
knn,1.0961,0.0000,0.4517,0.6537,0.4403
svd,1.1311,0.4517,0.0000,0.4348,0.2347
mf,1.2291,0.6537,0.4348,0.0000,0.4367
nn,1.1352,0.4403,0.2347,0.4367,0.0000
